# Step 1 — Gemma 4 E2B Baseline + Function Calling
**Tough Talks · Phase 1**

Goal: load `google/gemma-4-E2B-it`, confirm text generation works, then test **native** function calling using Gemma 4's chat-template tool protocol — the foundation every later step depends on.

In [1]:
# ── 0. Install / upgrade dependencies ────────────────────────────────────────
# Only bump transformers + accelerate. Do NOT bump torch on Colab/Kaggle:
# the pre-installed torch is paired with a torchvision built for the same
# CUDA version, and upgrading torch alone breaks that pairing
# (RuntimeError: PyTorch and torchvision were compiled with different CUDA
# major versions). If transformers later needs a newer torch, bump both
# torch and torchvision together.
#
# IMPORTANT: after running this cell once, restart the kernel before running
# anything else. pip upgrades don't take effect in an already-imported
# transformers module.

!pip install -q -U transformers accelerate
# Fallback if PyPI transformers does not yet include Gemma 4 — uncomment:
# !pip install -q --upgrade git+https://github.com/huggingface/transformers.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 105.0 MB/s eta 0:00:0000:010:01


In [3]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ───────────────────────
# Tries in order:
#   (a) cwd ancestor or a known cloud workspace path is already inside the repo
#       → if it's a git clone, refresh it from origin so you always run the
#         latest pushed code (no more `rm -rf` between iterations)
#   (b) clone REPO_URL into the workspace (Colab → /content, Kaggle → /kaggle/working)

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    """Force-update a shallow clone to match origin/HEAD."""
    if not (target / ".git").is_dir():
        return  # not a git checkout — assume locally edited, leave alone
    print(f"Refreshing {target} from origin")
    subprocess.run(
        ["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
        capture_output=True, check=False,
    )
    subprocess.run(
        ["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
        capture_output=True, check=False,
    )

REPO_ROOT = _scan_for_repo()

if REPO_ROOT is None:
    base = next(
        (b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
        pathlib.Path.cwd(),
    )
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(target)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"git clone failed (exit {result.returncode}).\n"
            "If the repo is private, make it public for the hackathon or upload "
            "the source manually to the workspace dir.\nstderr:\n" + result.stderr
        )
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

Cloning https://github.com/EhsanFarazmand/tough_talks.git -> /content/tough_talks
Repo root: /content/tough_talks


In [4]:
# ── 2. Imports ───────────────────────────────────────────────────────────────
import json

import torch

from backend.core._runtime import (
    DEFAULT_MODEL_ID,
    GenerationConfig,
    LoadConfig,
    ToolCallParseError,
    chat,
    load_model,
    load_tools,
    parse_tool_calls,
)

In [5]:
# ── 3. Configuration ─────────────────────────────────────────────────────────

MODEL_ID = DEFAULT_MODEL_ID                              # google/gemma-4-E2B-it
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

GEN_CFG = GenerationConfig(max_new_tokens=512, do_sample=False)

print(f"Model  : {MODEL_ID}")
print(f"Device : {DEVICE}")

Model  : google/gemma-4-E2B-it
Device : cuda


In [6]:
# ── 4. Load processor + model ────────────────────────────────────────────────
# Step 1 is text-only → AutoModelForCausalLM (default).
# Phase 2 (audio/transcription) will pass multimodal=True for AutoModelForMultimodalLM.

processor, model = load_model(LoadConfig(model_id=MODEL_ID))

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded ({n_params:.1f}B parameters, on {model.device})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded (5.1B parameters, on cuda:0)


In [7]:
# ── 5. Baseline generation ───────────────────────────────────────────────────
# `chat()` returns the raw decoded output (special tokens preserved). For a
# normal conversation we hand it to processor.parse_response — Gemma 4's
# official post-processor — which strips the protocol markers and returns
# clean text.

raw_baseline = chat(
    processor,
    model,
    messages=[
        {"role": "user", "content": "In one sentence, what is the most important skill in a difficult conversation?"}
    ],
    cfg=GEN_CFG,
)
baseline_reply = processor.parse_response(raw_baseline)

print("Baseline reply:")
print(baseline_reply)

Baseline reply:
{'role': 'assistant', 'content': "The most important skill in a difficult conversation is active and empathetic listening to truly understand the other person's perspective."}


In [8]:
# ── 6. Tool definitions (loaded from data/schemas/tools/) ────────────────────
# load_tools() returns the OpenAI function-tool format that Gemma 4's chat
# template expects in its `tools=` argument:
#   [{'type': 'function', 'function': {'name', 'description', 'parameters'}}]

TOOL_NAMES = ["analyze_emotion", "generate_whisper", "detect_talk_patterns"]
TOOLS      = load_tools(TOOL_NAMES)

print(f"Loaded {len(TOOLS)} tool definitions:")
for t in TOOLS:
    print(f"  - {t['function']['name']}")

Loaded 3 tool definitions:
  - analyze_emotion
  - generate_whisper
  - detect_talk_patterns


In [9]:
# ── 7. Function-calling helper ───────────────────────────────────────────────
# Native path: pass the tools to chat(), let Gemma 4's chat template emit
# tool declarations into the system block, parse the model's
# <|tool_call>...<tool_call|> response with parse_tool_calls.
# Returns the first parsed call, or {error, raw} on failure. Never raises.

def call_tool(user_request: str) -> dict:
    raw = chat(
        processor,
        model,
        messages=[{"role": "user", "content": user_request}],
        tools=TOOLS,
        cfg=GenerationConfig(max_new_tokens=256, do_sample=False),
    )
    try:
        calls = parse_tool_calls(raw)
    except ToolCallParseError as exc:
        return {"error": f"parse_tool_calls failed: {exc}", "raw": raw}
    if not calls:
        return {"error": "no tool call emitted", "raw": raw}
    result = dict(calls[0])
    if len(calls) > 1:
        result["_extra_calls"] = len(calls) - 1
    return result

In [10]:
# ── 8. Test: analyze_emotion ─────────────────────────────────────────────────
test_emotion = call_tool(
    'Analyze the emotion of this turn. The other person said: '
    '"Fine, do whatever you want, I don\'t even care anymore."'
)
print(json.dumps(test_emotion, indent=2))

{
  "name": "analyze_emotion",
  "arguments": {
    "speaker": "other",
    "text": "Fine, do whatever you want, I don't even care anymore."
  }
}


In [11]:
# ── 9. Test: generate_whisper ────────────────────────────────────────────────
test_whisper = call_tool(
    'Generate a coaching whisper. Conversation so far: '
    '["I just feel like you never listen.", "I do listen, you just never say anything clearly."]. '
    'Emotion state: {"tension": 0.85, "frustration": 0.7, "openness": 0.1}.'
)
print(json.dumps(test_whisper, indent=2))

{
  "name": "generate_whisper",
  "arguments": {
    "conversation_so_far": [
      "I just feel like you never listen.",
      "I do listen, you just never say anything clearly."
    ],
    "emotion_state": {
      "frustration": 0.7,
      "openness": 0.1,
      "tension": 0.85
    }
  }
}


In [12]:
# ── 10. Test: detect_talk_patterns ───────────────────────────────────────────
sample_transcript = (
    "User: I mean, I just feel like maybe, sort of, the project isn't going well? "
    "I'm sorry, I don't want to make this a big deal but I just feel like we're behind. "
    "Other: What do you mean behind? "
    "User: Oh no, not behind exactly — sorry, I shouldn't have said that. Never mind."
)
test_dna = call_tool(
    f'Detect talk patterns in this transcript: "{sample_transcript}"'
)
print(json.dumps(test_dna, indent=2))

{
  "name": "detect_talk_patterns",
  "arguments": {
    "transcript": "User: I mean, I just feel like maybe, sort of, the project isn't going well? I'm sorry, I don't want to make this a big deal but I just feel like we're behind. Other: What do you mean behind? User: Oh no, not behind exactly \u2014 sorry, I shouldn't have said that. Never mind."
  }
}


In [13]:
# ── 11. Step 1 results table ─────────────────────────────────────────────────
# Renders unconditionally, even if some tool calls failed — easier to see the
# full picture than to chase exceptions one at a time.

def _check(result: dict, expected_name: str, required_args: list[str]) -> tuple[bool, str]:
    if "error" in result:
        return False, f"error: {result['error']}"
    if result.get("name") != expected_name:
        return False, f"wrong tool: got {result.get('name')!r}"
    args = result.get("arguments") or {}
    missing = [a for a in required_args if a not in args]
    if missing:
        return False, f"missing args: {missing}"
    return True, "ok"


checks: list[tuple[str, bool, str]] = [
    ("model_loaded",        True,                 ""),
    ("baseline_generation", bool(baseline_reply), "non-empty reply" if baseline_reply else "empty"),
]
for name, result, required in [
    ("analyze_emotion",      test_emotion, ["speaker", "text"]),
    ("generate_whisper",     test_whisper, ["conversation_so_far"]),
    ("detect_talk_patterns", test_dna,     ["transcript"]),
]:
    ok, note = _check(result, name, required)
    checks.append((f"tool_call: {name}", ok, note))

print("=" * 60)
print("STEP 1 RESULTS")
print("=" * 60)
all_ok = True
for name, ok, note in checks:
    icon = "PASS" if ok else "FAIL"
    print(f"[{icon}]  {name:30s}  {note}")
    if not ok:
        all_ok = False

print()
print("OVERALL:", "READY FOR STEP 2" if all_ok else "FIX FAILURES ABOVE")

STEP 1 RESULTS
[PASS]  model_loaded                    
[PASS]  baseline_generation             non-empty reply
[PASS]  tool_call: analyze_emotion      ok
[PASS]  tool_call: generate_whisper     ok
[PASS]  tool_call: detect_talk_patterns  ok

OVERALL: READY FOR STEP 2
